# Phase 2/3 - end-to-end throughput: fused Triton attention in GPT-2

Measures the **whole-model tokens/sec lift** from replacing GPT-2's memory-bound (eager) attention with our fused Triton kernel, via a `scaled_dot_product_attention` monkeypatch. Correctness-gated: greedy text must match eager exactly.

**T4 GPU runtime, Run all.** For **Nsight** (`ncu`/`nsys`) profiling, see `NSIGHT.md` - that part needs a rented GPU with root, not Colab.

In [ ]:
!nvidia-smi --query-gpu=name --format=csv,noheader
import torch, triton
print("torch", torch.__version__, "| triton", triton.__version__)
assert torch.cuda.is_available(), "Set Runtime -> T4 GPU"

## 1. The kernel (generalized: prefill M==N and decode M==1)

In [ ]:
# --- generalized FlashAttention kernel (kernels/triton/fused_attention.py) ---
import torch
import triton
import triton.language as tl


# Fixed launch config (NOT @triton.autotune keyed on N_KV). Autotuning keyed on the
# KV length is pathological in a decode loop: N_KV grows by one every token, so the
# autotuner re-benchmarks all configs at every step, every layer — measured ~14x
# SLOWER end-to-end. A single fixed config avoids that and gives the real win
# (+~17% end-to-end vs eager attention at long context; see BENCHMARKS.md).
BLOCK_M = 64
BLOCK_N = 64
NUM_WARPS = 4
NUM_STAGES = 2


@triton.jit
def _attention_kernel(
    Q, K, V, Out,
    scale,
    stride_qb, stride_qh, stride_qm, stride_qd,
    stride_kb, stride_kh, stride_kn, stride_kd,
    stride_vb, stride_vh, stride_vn, stride_vd,
    stride_ob, stride_oh, stride_om, stride_od,
    H, N_Q, N_KV,
    D: tl.constexpr,
    CAUSAL: tl.constexpr,
    BLOCK_M: tl.constexpr,
    BLOCK_N: tl.constexpr,
):
    # Supports N_Q != N_KV (decode: 1 query vs a full KV cache). The queries are
    # the LAST N_Q positions of the sequence, so query row i has absolute position
    # q_offset + i, where q_offset = N_KV - N_Q. For prefill N_Q == N_KV (q_offset 0).
    pid_m = tl.program_id(0)
    pid_bh = tl.program_id(1)
    b = pid_bh // H
    h = pid_bh % H
    q_offset = N_KV - N_Q

    offs_m = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)
    offs_d = tl.arange(0, D)

    q_base = Q + b * stride_qb + h * stride_qh
    q = tl.load(q_base + offs_m[:, None] * stride_qm + offs_d[None, :] * stride_qd,
                mask=offs_m[:, None] < N_Q, other=0.0)   # [BLOCK_M, D]

    m_i = tl.full([BLOCK_M], -float("inf"), tl.float32)
    l_i = tl.zeros([BLOCK_M], tl.float32)
    acc = tl.zeros([BLOCK_M, D], tl.float32)

    # Causal: query at abs position (q_offset+m) attends keys <= that position.
    hi = q_offset + (pid_m + 1) * BLOCK_M if CAUSAL else N_KV
    k_base = K + b * stride_kb + h * stride_kh
    v_base = V + b * stride_vb + h * stride_vh

    for start_n in range(0, hi, BLOCK_N):
        offs_n = start_n + tl.arange(0, BLOCK_N)
        kt = tl.load(k_base + offs_d[:, None] * stride_kd + offs_n[None, :] * stride_kn,
                     mask=offs_n[None, :] < N_KV, other=0.0)   # [D, BLOCK_N], pre-transposed
        s = tl.dot(q, kt) * scale                             # [BLOCK_M, BLOCK_N]
        s = tl.where(offs_n[None, :] < N_KV, s, -float("inf"))
        if CAUSAL:
            s = tl.where((q_offset + offs_m[:, None]) >= offs_n[None, :], s, -float("inf"))

        m_new = tl.maximum(m_i, tl.max(s, 1))
        p = tl.exp(s - m_new[:, None])
        corr = tl.exp(m_i - m_new)
        l_i = l_i * corr + tl.sum(p, 1)

        v = tl.load(v_base + offs_n[:, None] * stride_vn + offs_d[None, :] * stride_vd,
                    mask=offs_n[:, None] < N_KV, other=0.0)     # [BLOCK_N, D]
        acc = acc * corr[:, None] + tl.dot(p.to(v.dtype), v)
        m_i = m_new

    acc = acc / l_i[:, None]
    o_base = Out + b * stride_ob + h * stride_oh
    tl.store(o_base + offs_m[:, None] * stride_om + offs_d[None, :] * stride_od,
             acc.to(Out.dtype.element_ty), mask=offs_m[:, None] < N_Q)


def fused_attention_bhsd(q, k, v, causal: bool = True):
    """Batched multi-head FlashAttention forward.

    q: [B, H, M, D], k/v: [B, H, N, D] with M <= N (M==N for prefill, M==1 for a
    single decode step against a length-N KV cache). Returns [B, H, M, D].
    """
    assert q.is_cuda and k.shape == v.shape
    assert q.shape[0] == k.shape[0] and q.shape[1] == k.shape[1] and q.shape[3] == k.shape[3]
    q, k, v = q.contiguous(), k.contiguous(), v.contiguous()
    B, H, M, D = q.shape
    N = k.shape[2]
    out = torch.empty_like(q)
    grid = (triton.cdiv(M, BLOCK_M), B * H)
    _attention_kernel[grid](
        q, k, v, out, D ** -0.5,
        q.stride(0), q.stride(1), q.stride(2), q.stride(3),
        k.stride(0), k.stride(1), k.stride(2), k.stride(3),
        v.stride(0), v.stride(1), v.stride(2), v.stride(3),
        out.stride(0), out.stride(1), out.stride(2), out.stride(3),
        H, M, N, D=D, CAUSAL=causal,
        BLOCK_M=BLOCK_M, BLOCK_N=BLOCK_N, num_warps=NUM_WARPS, num_stages=NUM_STAGES,
    )
    return out


def fused_attention(q, k, v, causal: bool = True):
    """Single-head attention via the Triton kernel (the stub's entry point)."""
    assert q.is_cuda and q.shape == k.shape == v.shape
    out = fused_attention_bhsd(q[None, None], k[None, None], v[None, None], causal)
    return out[0, 0]


## 2. Wire it into GPT-2 + correctness gate

Swap `F.scaled_dot_product_attention` for our kernel and confirm the generated text is identical to eager attention before measuring anything.

In [ ]:
import torch, torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL = "gpt2"
tok = AutoTokenizer.from_pretrained(MODEL)
if tok.pad_token_id is None: tok.pad_token = tok.eos_token

def load(impl):
    m = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float16,
                                             attn_implementation=impl).cuda().eval()
    return m

_orig_sdpa = F.scaled_dot_product_attention
def triton_sdpa(query, key, value, attn_mask=None, dropout_p=0.0, is_causal=False, scale=None, **kw):
    # Route SDPA through our kernel when safe; else fall back so output stays correct.
    if attn_mask is None and query.dtype in (torch.float16, torch.bfloat16) and query.shape[-1] == key.shape[-1]:
        from __main__ import fused_attention_bhsd  # defined in the kernel cell
        return fused_attention_bhsd(query, key, value, causal=bool(is_causal))
    return _orig_sdpa(query, key, value, attn_mask=attn_mask, dropout_p=dropout_p, is_causal=is_causal, scale=scale, **kw)

# ---- CORRECTNESS GATE: greedy output must match the eager baseline ----
m_eager = load("eager")
m_sdpa  = load("sdpa")
ids = tok("The meaning of life is", return_tensors="pt").input_ids.cuda()
with torch.inference_mode():
    out_eager = m_eager.generate(ids, max_new_tokens=30, do_sample=False, pad_token_id=tok.eos_token_id)
    F.scaled_dot_product_attention = triton_sdpa       # patch in our kernel
    out_triton = m_sdpa.generate(ids, max_new_tokens=30, do_sample=False, pad_token_id=tok.eos_token_id)
    F.scaled_dot_product_attention = _orig_sdpa        # unpatch
same = torch.equal(out_eager, out_triton)
print("eager :", tok.decode(out_eager[0, ids.shape[1]:]))
print("triton:", tok.decode(out_triton[0, ids.shape[1]:]))
assert same, "CORRECTNESS FAILED: triton-attention generation != eager. Paste this back."
print("\nCORRECTNESS GATE: PASS -- our kernel produces identical text to eager attention.")

## 3. End-to-end throughput: eager vs Triton vs SDPA

Long prompt + batch so attention is a real slice of the work. The lift is Triton vs the **eager** (memory-bound) baseline - the honest 'replaced a memory-bound attention kernel' comparison. SDPA is the ceiling.

In [ ]:
import time

def tps(model, patched, batch, prompt_len, new_tokens, reps=3):
    ids = torch.randint(0, 50000, (batch, prompt_len), device="cuda")
    if patched: F.scaled_dot_product_attention = triton_sdpa
    try:
        with torch.inference_mode():
            model.generate(ids, max_new_tokens=8, do_sample=False, pad_token_id=tok.eos_token_id)  # warmup
            torch.cuda.synchronize(); t0 = time.perf_counter()
            for _ in range(reps):
                model.generate(ids, max_new_tokens=new_tokens, do_sample=False, pad_token_id=tok.eos_token_id)
            torch.cuda.synchronize(); dt = time.perf_counter() - t0
    finally:
        F.scaled_dot_product_attention = _orig_sdpa
    return batch * new_tokens * reps / dt

# Long prompt + batch so PREFILL attention dominates (where the fused kernel wins).
# This regime measures ~+16-17% on an RTX 4090; short prompts give little/no lift.
BATCH, PROMPT_LEN, NEW = 16, 960, 16
eager_tps  = tps(m_eager, False, BATCH, PROMPT_LEN, NEW)
sdpa_tps   = tps(m_sdpa,  False, BATCH, PROMPT_LEN, NEW)
triton_tps = tps(m_sdpa,  True,  BATCH, PROMPT_LEN, NEW)

print(f"config: batch={BATCH} prompt_len={PROMPT_LEN} new_tokens={NEW}\n")
print(f"eager  attention : {eager_tps:8.1f} tok/s   (baseline, memory-bound)")
print(f"triton attention : {triton_tps:8.1f} tok/s   -> {triton_tps/eager_tps:.2f}x vs eager")
print(f"sdpa   attention : {sdpa_tps:8.1f} tok/s   (PyTorch fused, ceiling)")
print(f"\nEND-TO-END LIFT vs eager: {(triton_tps/eager_tps - 1)*100:+.1f}%")
print("Report the number YOU measured. Tune BATCH/PROMPT_LEN to explore the regime.")

## What to paste back

The **correctness** line and the **END-TO-END LIFT** number. If it's short of target, we tune the regime (bigger batch, longer prompt) or move to a larger model (TinyLlama) where attention is a bigger fraction - and report the true number for the config we use. Then run `NSIGHT.md` on a rented GPU for the `ncu`/`nsys` evidence.